### Overview:
Input = scans (720 scans over 180 degrees -> a scan every 0.25 degrees) (each scan has one distance associated with it)


### Features to test:
1. Size = angular width (number of returns). Will correspond to distance because closer people seem bigger/occupy more scans.
2. Position = scan distance. Can use position to filter out points that are too far away to be a part of a person's cluster.

In [1]:
# pip install tensorflow
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124


In [2]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# BRIAN'S PATH #
path = r"C:\Users\Brian\Desktop\School\ME4452 - Machine Learning for Engineers\Project\frog_11-36_12-43_train_val.h5"

# BEN'S PATH #
# path = r""

In [3]:
from sklearn.model_selection import train_test_split

with h5py.File(path, "r") as f:
    num_scans = len(f["scans"])

all_scan_indices = np.arange(num_scans)

# Reserve 20% for testing
train_val_indices, test_indices = train_test_split(
    all_scan_indices,
    test_size=0.20,
    random_state=42,
    shuffle=True,
)

# Split the remaining 80% into 60% training and 20% validation
train_indices, val_indices = train_test_split(
    train_val_indices,
    test_size=0.25,
    random_state=42,
    shuffle=True,
)

print(f"Train: {len(train_indices)} scans ({len(train_indices) / num_scans})")
print(f"Validation: {len(val_indices)} scans ({len(val_indices) / num_scans})")
print(f"Test: {len(test_indices)} scans ({len(test_indices) / num_scans})")

Train: 72237 scans (0.5999950164457291)
Validation: 24079 scans (0.19999833881524304)
Test: 24080 scans (0.20000664473902788)


In [4]:
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F

RNG = 42
BATCH_SIZE = 64
EPOCHS = 30
MAX_RANGE = 10.0
ANGLE_MIN = -np.pi / 2
ANGLE_MAX = np.pi / 2
TARGET_SIGMA_BEAMS = 2.0

np.random.seed(RNG)
torch.manual_seed(RNG)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

with h5py.File(path, "r") as f:
    NUM_BEAMS = f["scans"].shape[1]

print("Number of beams per scan:", NUM_BEAMS)

Using: cuda
Number of beams per scan: 720


In [5]:
model = nn.Sequential(
    nn.Conv1d(1, 32, kernel_size=7, padding=3),
    nn.ReLU(),

    nn.Conv1d(32, 64, kernel_size=5, padding=2),
    nn.ReLU(),

    nn.Conv1d(64, 32, kernel_size=3, padding=1),
    nn.ReLU(),

    # Output channel 0: confidence logits
    # Output channel 1: distance values before sigmoid
    nn.Conv1d(32, 2, kernel_size=1),
).to(device)

confidence_loss_fn = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([10.0], device=device)
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(model)

Sequential(
  (0): Conv1d(1, 32, kernel_size=(7,), stride=(1,), padding=(3,))
  (1): ReLU()
  (2): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
  (3): ReLU()
  (4): Conv1d(64, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (5): ReLU()
  (6): Conv1d(32, 2, kernel_size=(1,), stride=(1,))
)


In [ ]:
best_val_loss = float("inf")
best_model_state = None
patience = 5
epochs_without_improvement = 0

beam_ids = np.arange(NUM_BEAMS, dtype=np.float32)

with h5py.File(path, "r") as f:
    for epoch in range(EPOCHS):
        epoch_losses = {}

        for phase, split_indices in [
            ("train", train_indices),
            ("validation", val_indices),
        ]:
            is_training = phase == "train"
            model.train(is_training)

            if is_training:
                np.random.shuffle(split_indices)

            total_loss = 0.0
            total_scans = 0

            batch_starts = range(0, len(split_indices), BATCH_SIZE)

            for batch_start in batch_starts:
                batch_indices = split_indices[
                    batch_start : batch_start + BATCH_SIZE
                ]
                current_batch_size = len(batch_indices)

                x_batch = np.zeros(
                    (current_batch_size, 1, NUM_BEAMS), dtype=np.float32
                )
                confidence_batch = np.zeros(
                    (current_batch_size, NUM_BEAMS), dtype=np.float32
                )
                distance_batch = np.zeros(
                    (current_batch_size, NUM_BEAMS), dtype=np.float32
                )
                distance_mask_batch = np.zeros(
                    (current_batch_size, NUM_BEAMS), dtype=np.float32
                )

                # Load scans and create their annotation targets
                for row, scan_idx in enumerate(batch_indices):
                    scan_idx = int(scan_idx)

                    ranges = np.asarray(
                        f["scans"][scan_idx], dtype=np.float32
                    )
                    ranges = np.nan_to_num(
                        ranges,
                        nan=MAX_RANGE,
                        posinf=MAX_RANGE,
                        neginf=0.0,
                    )
                    ranges = np.clip(ranges, 0.0, MAX_RANGE) / MAX_RANGE
                    x_batch[row, 0, :] = ranges

                    start = int(f["circle_idx"][scan_idx])
                    count = int(f["circle_num"][scan_idx])
                    people = f["circles"][start : start + count]

                    for person in people:
                        # FROG row:
                        # [x, y, radius, distance, angle, angular_radius]
                        person_distance = float(person[3])
                        person_angle = float(person[4])

                        beam = int(round(
                            (person_angle - ANGLE_MIN)
                            / (ANGLE_MAX - ANGLE_MIN)
                            * (NUM_BEAMS - 1)
                        ))

                        if not 0 <= beam < NUM_BEAMS:
                            continue

                        peak = np.exp(
                            -0.5
                            * (
                                (beam_ids - beam)
                                / TARGET_SIGMA_BEAMS
                            ) ** 2
                        )
                        confidence_batch[row] = np.maximum(
                            confidence_batch[row], peak
                        )

                        distance_batch[row, beam] = np.clip(
                            person_distance / MAX_RANGE, 0.0, 1.0
                        )
                        distance_mask_batch[row, beam] = 1.0

                scans = torch.from_numpy(x_batch).to(device)
                conf_target = torch.from_numpy(confidence_batch).to(device)
                dist_target = torch.from_numpy(distance_batch).to(device)
                dist_mask = torch.from_numpy(distance_mask_batch).to(device)

                with torch.set_grad_enabled(is_training):
                    outputs = model(scans)

                    conf_logits = outputs[:, 0, :]
                    dist_prediction = torch.sigmoid(outputs[:, 1, :])

                    confidence_loss = confidence_loss_fn(
                        conf_logits, conf_target
                    )

                    distance_errors = F.smooth_l1_loss(
                        dist_prediction,
                        dist_target,
                        reduction="none",
                    )
                    distance_loss = (
                        (distance_errors * dist_mask).sum()
                        / dist_mask.sum().clamp(min=1.0)
                    )

                    loss = confidence_loss + distance_loss

                    if is_training:
                        optimizer.zero_grad()
                        loss.backward()
                        optimizer.step()

                total_loss += loss.item() * current_batch_size
                total_scans += current_batch_size

            epoch_losses[phase] = total_loss / total_scans

        train_loss = epoch_losses["train"]
        val_loss = epoch_losses["validation"]

        print(
            f"Epoch {epoch + 1}/{EPOCHS} complete | "
            f"train loss: {train_loss:.4f} | "
            f"validation loss: {val_loss:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

            if epochs_without_improvement >= patience:
                print("Early stopping.")
                break

model.load_state_dict(best_model_state)
torch.save(model.state_dict(), "frog_person_detector.pt")

print(f"Best validation loss: {best_val_loss}")

Epoch 1/30 complete | train loss: 0.5833 | validation loss: 0.5448
Epoch 2/30 complete | train loss: 0.5344 | validation loss: 0.5198
Epoch 3/30 complete | train loss: 0.5168 | validation loss: 0.5142
Epoch 4/30 complete | train loss: 0.5042 | validation loss: 0.5076
Epoch 5/30 complete | train loss: 0.4964 | validation loss: 0.4883
Epoch 6/30 complete | train loss: 0.4849 | validation loss: 0.4740
Epoch 7/30 complete | train loss: 0.4775 | validation loss: 0.4938
Epoch 8/30 complete | train loss: 0.4733 | validation loss: 0.4967
Epoch 9/30 complete | train loss: 0.4693 | validation loss: 0.4643
Epoch 10/30 complete | train loss: 0.4668 | validation loss: 0.4595
Epoch 11/30 complete | train loss: 0.4627 | validation loss: 0.4700
Epoch 12/30 complete | train loss: 0.4624 | validation loss: 0.4735
Epoch 13/30 complete | train loss: 0.4594 | validation loss: 0.4591
Epoch 14/30 complete | train loss: 0.4588 | validation loss: 0.4557
Epoch 15/30 complete | train loss: 0.4569 | validation lo

In [9]:
from scipy.optimize import linear_sum_assignment
from scipy.signal import find_peaks

CONFIDENCE_THRESHOLD = 0.70  # Tune on validation data, then keep fixed
MIN_PEAK_SPACING = 10         # Minimum separation in beams
MATCH_TOLERANCE = 0.5        # Maximum matching error in meters

beam_angles = np.linspace(ANGLE_MIN, ANGLE_MAX, NUM_BEAMS)

true_positives = 0
false_positives = 0
false_negatives = 0
xy_errors = []

model.eval()

with h5py.File(path, "r") as f:
    with torch.no_grad():
        for scan_idx in test_indices:
            scan_idx = int(scan_idx)

            # Prepare the scan and predict confidence and distance per beam
            ranges = np.asarray(f["scans"][scan_idx], dtype=np.float32)
            ranges = np.nan_to_num(
                ranges, nan=MAX_RANGE, posinf=MAX_RANGE, neginf=0.0
            )
            ranges = np.clip(ranges, 0.0, MAX_RANGE) / MAX_RANGE

            scan_tensor = torch.from_numpy(ranges).unsqueeze(0).unsqueeze(0)
            scan_tensor = scan_tensor.to(device)

            output = model(scan_tensor)
            confidence = torch.sigmoid(output[0, 0]).cpu().numpy()
            predicted_distance = (
                torch.sigmoid(output[0, 1]).cpu().numpy() * MAX_RANGE
            )

            # Find peaks in the confidence output
            peaks, _ = find_peaks(
                confidence,
                height=CONFIDENCE_THRESHOLD,
                distance=MIN_PEAK_SPACING,
            )

            # Convert predicted angle and distance to x/y coordinates
            predicted_points = []
            for beam in peaks:
                angle = beam_angles[beam]
                distance = predicted_distance[beam]
                x = distance * np.cos(angle)
                y = distance * np.sin(angle)
                predicted_points.append([x, y])

            predicted_points = np.asarray(
                predicted_points, dtype=np.float32
            ).reshape(-1, 2)

            # Get the ground-truth person centers for this scan
            start = int(f["circle_idx"][scan_idx])
            count = int(f["circle_num"][scan_idx])
            people = f["circles"][start : start + count]
            true_points = np.asarray(people[:, :2], dtype=np.float32).reshape(-1, 2)

            num_predicted = len(predicted_points)
            num_true = len(true_points)
            matches = []

            # One-to-one matching, allowing at most 0.5 m error
            if num_predicted > 0 and num_true > 0:
                distances = np.linalg.norm(
                    predicted_points[:, None, :] - true_points[None, :, :],
                    axis=2,
                )

                # Dummy assignments allow predictions or annotations to remain unmatched
                size = num_predicted + num_true
                large_cost = 1e6
                unmatched_cost = MATCH_TOLERANCE + 1e-3
                cost = np.full((size, size), large_cost, dtype=np.float32)

                valid_matches = distances <= MATCH_TOLERANCE
                cost[:num_predicted, :num_true] = np.where(
                    valid_matches, distances, large_cost
                )

                for i in range(num_predicted):
                    cost[i, num_true + i] = unmatched_cost

                for j in range(num_true):
                    cost[num_predicted + j, j] = unmatched_cost

                cost[num_predicted:, num_true:] = 0.0

                rows, cols = linear_sum_assignment(cost)

                for row, col in zip(rows, cols):
                    if (
                        row < num_predicted
                        and col < num_true
                        and distances[row, col] <= MATCH_TOLERANCE
                    ):
                        matches.append((row, col))

            true_positives += len(matches)
            false_positives += num_predicted - len(matches)
            false_negatives += num_true - len(matches)

            for pred_i, true_i in matches:
                xy_errors.append(predicted_points[pred_i] - true_points[true_i])


# Detection metrics at the selected confidence threshold and 0.5 m tolerance
precision = true_positives / max(true_positives + false_positives, 1)
recall = true_positives / max(true_positives + false_negatives, 1)

print(f"True positives:  {true_positives}")
print(f"False positives: {false_positives}")
print(f"False negatives: {false_negatives}")
print(f"Precision:       {precision:.3f}")
print(f"Recall:          {recall:.3f}")

# Position errors for correctly matched detections
xy_errors = np.asarray(xy_errors, dtype=np.float32).reshape(-1, 2)

if len(xy_errors) > 0:
    euclidean_errors = np.linalg.norm(xy_errors, axis=1)

    print(f"Mean x error (absolute): {np.mean(np.abs(xy_errors[:, 0]))} m")
    print(f"Mean y error (absolute): {np.mean(np.abs(xy_errors[:, 1]))} m")
    print(f"Mean position error:     {np.mean(euclidean_errors)} m")
    print(f"Median position error:   {np.median(euclidean_errors)} m")
    print(f"Position error RMSE:     {np.sqrt(np.mean(euclidean_errors**2))} m")
else:
    print("No predictions matched annotations within the tolerance.")

True positives:  55733
False positives: 80291
False negatives: 38533
Precision:       0.410
Recall:          0.591
Mean x error (absolute): 0.11800415813922882 m
Mean y error (absolute): 0.08582909405231476 m
Mean position error:     0.16137665510177612 m
Median position error:   0.13593582808971405 m
Position error RMSE:     0.1965891569852829 m
